# HerWILL Datathon: ensemble weight tuning (run once)

Loads the six `{tag}_final/` checkpoints from the memorization training notebooks, runs them on
**`validation.csv`**, fits every post-processing weight on it, and writes **`ensemble_bundle/`**.
That bundle is all `ensemble_inference_prod.ipynb` needs besides the checkpoints: it does **no
fitting, tuning or training**.

**Why the weights are fitted on validation and not on train:** the models were trained to
memorize `train.csv`, so their train predictions are close to 100% correct and useless for
calibration. Validation is the only data that shows how they behave on unseen text.

**What is tuned (in pipeline order), with ideas taken from `exploration.ipynb`:**

| # | stage | fitted weights | why |
|---|---|---|---|
| 1 | **temperature + blend** | one temperature `T` per model (validation log-loss), blend weights (random search on the simplex, falls back to equal weights if the gain is < 0.002) | memorized models are over-confident, so their probabilities have to be softened before averaging means anything |
| 2 | **kNN soft mix** | `alpha`, `k`, `min_sim` | memory of the training data: similarity-weighted labels of the nearest char n-gram TF-IDF neighbours in train, mixed into the probabilities |
| 3 | **class weights** | 3 multipliers `W` (random + coordinate search) | fixes class imbalance for macro-F1 (class 2 is ~12% of rows but a third of the score) |
| 4 | **exact-match override** | none | a text seen verbatim in train gets its (majority) train label |
| 5 | **kNN hard override** | `tau_hard` | a near-duplicate (cosine ≥ `tau_hard`) of a train text takes that text's label |

**Honesty check:** the whole tuning procedure runs inside a 5-fold CV over validation. Every stage's
held-out gain is reported, and a stage with a negative mean held-out gain is **switched off
automatically** before the final fit on all of validation.

For production, the exact-match and kNN memories are rebuilt on **train + validation**, so
production remembers the validation labels too.

## 1. Config

In [ ]:
# (tag, preprocessing family). Tag must match the checkpoint folder name "{tag}_final".
MEMBERS = [
    ("hatebert",         "multilingual"),
    ("twitter_xlmr",     "multilingual"),
    ("xlmr_large",       "multilingual"),
    ("banglabert",       "bangla"),
    ("banglishbert",     "bangla"),
    ("banglabert_large", "bangla"),
]

TRAIN_PATH = None      # e.g. "/kaggle/input/herwill-data/train.csv"   (same file the models trained on)
VAL_PATH   = None      # e.g. "/kaggle/input/herwill-data/validation.csv"  (must have labels)
CKPT_ROOTS = ["/kaggle/input", "/kaggle/working", "."]   # searched for "{tag}_final/head_config.json"


class TUNE:
    SEED            = 42
    EVAL_BS         = 256
    CACHE_LOGITS    = True                      # reuse val logits on rerun (skips model inference)

    TEMP_GRID       = np.exp(np.linspace(np.log(0.05), np.log(50), 161))
    BLEND_N_RANDOM  = 3000
    BLEND_MIN_GAIN  = 0.002                     # below this, use equal blend weights

    KNN_K_MAX       = 20
    KNN_K_GRID      = [5, 10, 20]
    KNN_MIN_SIM     = [0.3, 0.4, 0.5, 0.6, 0.7]
    KNN_ALPHA       = [round(a, 2) for a in np.arange(0.0, 0.65, 0.05)]
    KNN_MIN_GAIN    = 0.001
    TAU_GRID        = [None, 0.99, 0.97, 0.95, 0.92, 0.90, 0.87, 0.85, 0.80]

    CW_N_RANDOM     = 4000

    CV_FOLDS        = 5
    AUTO_DISABLE    = True                      # switch off stages whose CV held-out gain < 0
    PROD_MEMORY_INCLUDES_VAL = True             # production lookup/kNN = train + validation


BUNDLE_DIR = OUT_DIR / "ensemble_bundle"
CACHE_DIR = OUT_DIR / "ensemble_cache"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True); CACHE_DIR.mkdir(parents=True, exist_ok=True)
random.seed(TUNE.SEED); np.random.seed(TUNE.SEED)

## 2. Environment

In [ ]:
import os, re, gc, json, time, random, warnings, subprocess, sys
from pathlib import Path

# Everything loads from local checkpoint folders; never touch the HF Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoConfig
import transformers

warnings.filterwarnings("ignore")

N_CLASSES = 3
CLASS_NAMES = {0: "Explicitly Toxic", 1: "Subtly Toxic", 2: "Non-toxic"}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    AMP_DTYPE = None

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| amp", AMP_DTYPE)

## 3. Preprocessing (identical to training)

In [ ]:
# Preprocessing: must be byte-for-byte what the training notebooks did.
#   "multilingual" family (hatebert / twitter_xlmr / xlmr_large) -> _base_clean only
#   "bangla" family (banglabert / banglishbert / banglabert_large) -> csebuetnlp normalizer, then _base_clean
try:
    from normalizer import normalize as bn_normalize
except ImportError:
    # offline: wheel from the offline-assets dataset; online fallback: GitHub
    whl = next(iter(sorted(Path("/kaggle/input").rglob("normalizer-*.whl"))), None) \
        if Path("/kaggle/input").exists() else None
    if whl is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                        "--find-links", str(whl.parent), "normalizer"], check=False)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/csebuetnlp/normalizer"], check=False)
    try:
        from normalizer import normalize as bn_normalize
    except ImportError:
        bn_normalize = None
BN_NORMALIZER_AVAILABLE = bn_normalize is not None
print("csebuetnlp normalizer available:", BN_NORMALIZER_AVAILABLE)

import unicodedata as ud

ZW_RE      = re.compile(r"[​‎‏﻿­⁠]")
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"(?<!\w)@\w+")
ELONG_RE   = re.compile(r"([A-Za-z])\1{2,}")
PUNCT_RE   = re.compile(r"([!?.,\-*])\1{2,}")
MAX_CHARS  = 2000


def _base_clean(s):
    s = ud.normalize("NFC", s)
    s = ZW_RE.sub("", s)
    s = URL_RE.sub(" [URL] ", s)
    s = MENTION_RE.sub(" [USER] ", s)
    s = ELONG_RE.sub(r"\1\1", s)
    s = PUNCT_RE.sub(r"\1\1", s)
    s = s[:MAX_CHARS]
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else "[EMPTY]"


def clean_multilingual(s):
    return _base_clean(str(s))


def clean_bangla(s):
    s = str(s)
    if bn_normalize is not None:
        try:
            s = bn_normalize(s)
        except Exception:
            pass
    return _base_clean(s)


PREPROCESSORS = {"multilingual": clean_multilingual, "bangla": clean_bangla}


def norm_key(s):
    # Key for the exact-match lookup and the kNN memory. It does not depend on the Bangla
    # normalizer, so it behaves the same whether or not that package is installed.
    return clean_multilingual(s).lower()


def load_csv(path, need_labels):
    df = pd.read_csv(path)
    text_col = "text" if "text" in df.columns else next(
        c for c in df.columns if c.lower() in ("text", "sentence", "comment", "content"))
    label_col = "y" if "y" in df.columns else next(
        (c for c in df.columns if c.lower() in ("y", "label", "labels", "target", "class")), None)
    if need_labels and label_col is None:
        raise ValueError(f"{path} has no label column")
    out = pd.DataFrame({
        "id": df["id"] if "id" in df.columns else np.arange(len(df)),
        "text": df[text_col].fillna("").astype(str),
    })
    if label_col is not None and need_labels:
        out["y"] = df[label_col]
        out = out.dropna(subset=["y"])
        out["y"] = out["y"].astype(int)
    print(f"{Path(path).name}: {len(out)} rows")
    return out.reset_index(drop=True)

## 4. Model loading & inference

In [ ]:
# Same architecture as the training notebooks, rebuilt from the LOCAL checkpoint folder
# (config.json + tokenizer + head_config.json + model_state.pt), so no internet is needed.
class BigHeadClassifier(nn.Module):
    def __init__(self, ckpt_dir):
        super().__init__()
        meta = json.load(open(Path(ckpt_dir) / "head_config.json"))
        config = AutoConfig.from_pretrained(ckpt_dir)
        extra = {"add_pooling_layer": False} if config.model_type in ("bert", "roberta", "xlm-roberta") else {}
        self.encoder = AutoModel.from_config(config, **extra)
        H = config.hidden_size
        layers, d_in = [], 2 * H
        for d_out in meta["hidden_sizes"]:
            layers += [nn.Linear(d_in, d_out), nn.LayerNorm(d_out), nn.GELU()]
            d_in = d_out
        layers.append(nn.Linear(d_in, meta["n_classes"]))
        self.head = nn.Sequential(*layers)
        self.meta = meta

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        h = self.encoder(**kw).last_hidden_state
        cls = h[:, 0]
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1.0)
        return self.head(torch.cat([cls, mean], dim=-1).float())


def find_checkpoint(tag, roots):
    for r in roots:
        r = Path(r)
        if not r.exists():
            continue
        for hc in r.rglob("head_config.json"):
            if hc.parent.name == f"{tag}_final":
                return hc.parent
    return None


def load_member(ckpt_dir):
    model = BigHeadClassifier(ckpt_dir)
    sd = torch.load(Path(ckpt_dir) / "model_state.pt", map_location="cpu", weights_only=True)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    # position_ids buffers differ between transformers versions; anything else is a real mismatch
    missing = [k for k in missing if not k.endswith("position_ids")]
    unexpected = [k for k in unexpected if not k.endswith("position_ids")]
    if missing or unexpected:
        raise RuntimeError(f"state_dict mismatch in {ckpt_dir}: missing={missing[:5]} unexpected={unexpected[:5]}")
    tok = AutoTokenizer.from_pretrained(ckpt_dir)
    return model.to(DEVICE).eval(), tok


@torch.no_grad()
def predict_logits(model, tok, texts, max_len, bs=256):
    enc = tok(list(texts), truncation=True, max_length=max_len)
    keys = list(enc.keys())
    order = np.argsort([len(x) for x in enc["input_ids"]], kind="stable")   # length-sorted = little padding
    out = np.zeros((len(texts), N_CLASSES), dtype=np.float32)
    for s in range(0, len(order), bs):
        sel = order[s:s + bs]
        batch = tok.pad([{k: enc[k][i] for k in keys} for i in sel], return_tensors="pt")
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE or torch.float32, enabled=AMP_DTYPE is not None):
            z = model(**batch)
        out[sel] = z.float().cpu().numpy()
    return out


_clean_cache = {}

def run_member(tag, family, ckpt_dir, raw_texts, bs=256):
    ck = (family, id(raw_texts))
    if ck not in _clean_cache:
        _clean_cache[ck] = [PREPROCESSORS[family](t) for t in raw_texts]
    t0 = time.time()
    model, tok = load_member(ckpt_dir)
    n_params = sum(p.numel() for p in model.parameters())
    logits = predict_logits(model, tok, _clean_cache[ck], model.meta["max_len"], bs)
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"  {tag:<18} {n_params/1e6:7.1f}M params  {len(raw_texts)} rows  {time.time()-t0:.0f}s")
    return logits, int(n_params)

## 5. Post-processing (`final_predict`, shared with production)

In [ ]:
# Post-processing: the ONE function that turns member logits into labels (final_predict).
# Tuning calls it with candidate params; production calls it with the saved params.
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib


def softmax_T(z, T):
    z = z / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def macro_f1(y, pred):
    cm = np.bincount(y * N_CLASSES + pred, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros(N_CLASSES), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros(N_CLASSES), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(N_CLASSES), where=(prec + rec) > 0)
    present = (cm.sum(0) + cm.sum(1)) > 0
    return float(f1[present].mean())


def build_lookup(keys, ys):
    # exact-text memory: majority label per normalised text, ties dropped (from exploration.ipynb)
    df = pd.DataFrame({"key": list(keys), "y": list(ys)})
    out = {}
    for key, g in df.groupby("key", sort=False):
        vc = g["y"].value_counts()
        if len(vc) > 1 and vc.iloc[0] == vc.iloc[1]:
            continue
        out[key] = int(vc.index[0])
    return out


def exact_labels(keys, lookup):
    return np.array([lookup.get(k, -1) for k in keys], dtype=np.int64)


class KnnIndex:
    """Char n-gram TF-IDF memory of labelled texts: cosine nearest neighbours of each query."""

    def __init__(self, ngram_range=(2, 5), min_df=2, max_features=500_000):
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=ngram_range, min_df=min_df,
                                   max_features=max_features, sublinear_tf=True,
                                   lowercase=False, dtype=np.float32)
        self.MT, self.labels = None, None

    def fit(self, keys, labels):
        self.MT = self.vec.fit_transform(list(keys)).T.tocsr()
        self.labels = np.asarray(labels, dtype=np.int64)
        return self

    def topk(self, keys, k, chunk=512):
        Q = self.vec.transform(list(keys))
        k = min(k, len(self.labels))
        sims = np.zeros((Q.shape[0], k), dtype=np.float32)
        idx = np.zeros((Q.shape[0], k), dtype=np.int64)
        for s in range(0, Q.shape[0], chunk):
            S = (Q[s:s + chunk] @ self.MT).toarray()
            part = np.argpartition(-S, k - 1, axis=1)[:, :k]
            ps = np.take_along_axis(S, part, 1)
            o = np.argsort(-ps, axis=1)
            idx[s:s + chunk] = np.take_along_axis(part, o, 1)
            sims[s:s + chunk] = np.take_along_axis(ps, o, 1)
        return sims, self.labels[idx]

    def save(self, path):
        joblib.dump({"vec": self.vec, "MT": self.MT, "labels": self.labels}, path, compress=3)

    @classmethod
    def load(cls, path):
        d = joblib.load(path)
        obj = cls.__new__(cls)
        obj.vec, obj.MT, obj.labels = d["vec"], d["MT"], d["labels"]
        return obj


def knn_probs(sims, labels, min_sim):
    w = np.where(sims >= min_sim, sims, 0.0)
    pk = np.stack([(w * (labels == c)).sum(1) for c in range(N_CLASSES)], axis=1)
    has = pk.sum(1) > 0
    pk[has] /= pk[has].sum(1, keepdims=True)
    return pk, has


def final_predict(member_logits, knn_sims, knn_labels, exact, params):
    """member probs (temperature) -> weighted blend -> kNN soft mix -> class weights -> argmax
       -> kNN hard override (near-duplicate) -> exact-match override."""
    mem = params["members"]
    w = np.array([m["weight"] for m in mem], dtype=np.float64)
    w = w / w.sum()
    p = sum(wi * softmax_T(member_logits[m["tag"]], m["temperature"]) for wi, m in zip(w, mem))

    kn = params["knn"]
    if kn["alpha"] > 0:
        pk, has = knn_probs(knn_sims[:, :kn["k"]], knn_labels[:, :kn["k"]], kn["min_sim"])
        p = np.where(has[:, None], (1 - kn["alpha"]) * p + kn["alpha"] * pk, p)

    p = p * np.asarray(params["class_weights"], dtype=np.float64)
    p = p / p.sum(1, keepdims=True)
    pred = p.argmax(1)
    source = np.full(len(pred), "model", dtype=object)

    if kn["tau_hard"] is not None:
        m = knn_sims[:, 0] >= kn["tau_hard"]
        pred[m] = knn_labels[m, 0]; source[m] = "knn_hard"
    if params["exact_match"]:
        m = exact >= 0
        pred[m] = exact[m]; source[m] = "exact"
    return pred, p, source

## 6. Load data & build the train memory (exact lookup + kNN)

In [ ]:
def find_file(explicit, name):
    if explicit is not None:
        return Path(explicit)
    hits = sorted(Path("/kaggle/input").rglob(name)) if Path("/kaggle/input").exists() else []
    hits += [p for p in (Path(name), Path("data") / name) if p.exists()]
    if not hits:
        raise FileNotFoundError(f"{name} not found; set it in the config cell")
    return hits[0]


train = load_csv(find_file(TRAIN_PATH, "train.csv"), need_labels=True)
val   = load_csv(find_file(VAL_PATH, "validation.csv"), need_labels=True)
y_val = val["y"].values

train_keys = train["text"].map(norm_key).tolist()
val_keys   = val["text"].map(norm_key).tolist()

# Tuning memory = train only (validation is the query, so no self-matches).
lookup_tr = build_lookup(train_keys, train["y"])
exact_val = exact_labels(val_keys, lookup_tr)
hit = exact_val >= 0
print(f"exact matches val->train: {hit.sum()} / {len(val)}"
      + (f"  (label agreement {np.mean(exact_val[hit] == y_val[hit]):.3f})" if hit.any() else ""))

t0 = time.time()
knn_tr = KnnIndex().fit(train_keys, train["y"])
val_sims, val_nlab = knn_tr.topk(val_keys, TUNE.KNN_K_MAX)
print(f"kNN over train built + queried in {time.time()-t0:.0f}s; "
      f"top-1 sim median {np.median(val_sims[:, 0]):.3f}, >=0.9: {(val_sims[:, 0] >= 0.9).mean():.2%}")
for th in [0.8, 0.9, 0.95]:
    m = val_sims[:, 0] >= th
    if m.any():
        print(f"  top-1 sim >= {th}: {m.sum():5d} rows, neighbour label accuracy {np.mean(val_nlab[m, 0] == y_val[m]):.3f}")

## 7. Member logits on validation

In [ ]:
VAL_LOGITS, MEMBER_PARAMS, MEMBER_FAMILY = {}, {}, {}
raw_val = val["text"].tolist()
for tag, family in MEMBERS:
    cache = CACHE_DIR / f"val_logits_{tag}.npy"
    meta_cache = CACHE_DIR / f"val_logits_{tag}.json"
    if TUNE.CACHE_LOGITS and cache.exists() and meta_cache.exists():
        z = np.load(cache)
        if z.shape[0] == len(val):
            VAL_LOGITS[tag] = z
            MEMBER_PARAMS[tag] = json.load(open(meta_cache))["params"]
            MEMBER_FAMILY[tag] = family
            print(f"  {tag:<18} loaded cached logits")
            continue
    ckpt = find_checkpoint(tag, CKPT_ROOTS)
    if ckpt is None:
        print(f"  [MISSING] {tag}: no {tag}_final/ under {CKPT_ROOTS} -> left out of the ensemble")
        continue
    z, n_params = run_member(tag, family, ckpt, raw_val, TUNE.EVAL_BS)
    VAL_LOGITS[tag], MEMBER_PARAMS[tag], MEMBER_FAMILY[tag] = z, n_params, family
    np.save(cache, z); json.dump({"params": n_params, "ckpt": str(ckpt)}, open(meta_cache, "w"))

TAGS = list(VAL_LOGITS)
assert TAGS, "no ensemble member found"
print("\nper-model validation macro-F1 (plain argmax):")
for t in TAGS:
    print(f"  {t:<18} {macro_f1(y_val, VAL_LOGITS[t].argmax(1)):.4f}")

## 8. Tuning procedure

`tune_all` fits stages 1 to 5 in order on whatever rows it is given. It is used both inside CV and for the final fit.

In [ ]:
STAGES = ["temp_blend", "knn_soft", "class_w", "exact", "knn_hard"]


def neutral_params(tags):
    return {
        "members": [{"tag": t, "family": MEMBER_FAMILY[t], "temperature": 1.0, "weight": 1.0,
                     "params": MEMBER_PARAMS[t]} for t in tags],
        "knn": {"k": 10, "min_sim": 0.5, "alpha": 0.0, "tau_hard": None},
        "class_weights": [1.0] * N_CLASSES,
        "exact_match": False,
    }


def neutralize(params, off):
    p = json.loads(json.dumps(params))
    if "temp_blend" in off:
        for m in p["members"]:
            m["temperature"], m["weight"] = 1.0, 1.0
    if "knn_soft" in off: p["knn"]["alpha"] = 0.0
    if "class_w" in off:  p["class_weights"] = [1.0] * N_CLASSES
    if "exact" in off:    p["exact_match"] = False
    if "knn_hard" in off: p["knn"]["tau_hard"] = None
    return p


def score(params, Z, S, L, E, y):
    return macro_f1(y, final_predict(Z, S, L, E, params)[0])


def fit_temperature(z, y):
    idx = np.arange(len(y))
    nll = [-np.log(np.clip(softmax_T(z, T)[idx, y], 1e-12, None)).mean() for T in TUNE.TEMP_GRID]
    return float(TUNE.TEMP_GRID[int(np.argmin(nll))])


def fit_blend(P, y, seed):
    tags = list(P)
    stack = np.stack([P[t] for t in tags])                   # (M, n, C)
    f = lambda w: macro_f1(y, np.tensordot(w / w.sum(), stack, 1).argmax(1))
    equal = np.ones(len(tags))
    best_w, best = equal, f(equal)
    base = best
    if len(tags) > 1:
        for w in np.random.default_rng(seed).dirichlet(np.ones(len(tags)), size=TUNE.BLEND_N_RANDOM):
            s = f(w)
            if s > best:
                best, best_w = s, w
    if best - base < TUNE.BLEND_MIN_GAIN:
        best_w = equal
    return dict(zip(tags, (best_w / best_w.sum() * len(tags)).tolist()))


def fit_class_weights(p, y, seed):
    f = lambda w: macro_f1(y, (p * w).argmax(1))
    best_w = np.ones(N_CLASSES); best = f(best_w)
    for w in np.random.default_rng(seed).dirichlet(np.ones(N_CLASSES), size=TUNE.CW_N_RANDOM) * N_CLASSES:
        s = f(w)
        if s > best:
            best, best_w = s, w
    for _ in range(4):                                         # coordinate refinement
        for c in range(N_CLASSES):
            for mult in np.linspace(0.8, 1.25, 46):
                w = best_w.copy(); w[c] *= mult
                s = f(w)
                if s > best:
                    best, best_w = s, w
    return (best_w / best_w.mean()).tolist()


def tune_all(Z, S, L, E, y, disabled=frozenset(), seed=0):
    params = neutral_params(list(Z))

    if "temp_blend" not in disabled:
        T = {t: fit_temperature(Z[t], y) for t in Z}
        w = fit_blend({t: softmax_T(Z[t], T[t]) for t in Z}, y, seed)
        for m in params["members"]:
            m["temperature"], m["weight"] = T[m["tag"]], w[m["tag"]]

    if "knn_soft" not in disabled:
        base = score(params, Z, S, L, E, y)
        best, best_cfg = base, None
        for k in TUNE.KNN_K_GRID:
            for ms in TUNE.KNN_MIN_SIM:
                for a in TUNE.KNN_ALPHA[1:]:
                    params["knn"].update(k=k, min_sim=ms, alpha=a)
                    s = score(params, Z, S, L, E, y)
                    if s > best:
                        best, best_cfg = s, (k, ms, a)
        if best_cfg is None or best - base < TUNE.KNN_MIN_GAIN:
            params["knn"].update(k=10, min_sim=0.5, alpha=0.0)
        else:
            params["knn"].update(k=best_cfg[0], min_sim=best_cfg[1], alpha=best_cfg[2])

    if "class_w" not in disabled:
        _, p, _ = final_predict(Z, S, L, E, params)
        params["class_weights"] = fit_class_weights(p, y, seed)

    if "exact" not in disabled:
        params["exact_match"] = True

    if "knn_hard" not in disabled:
        best, best_tau = -1, None
        for tau in TUNE.TAU_GRID:
            params["knn"]["tau_hard"] = tau
            s = score(params, Z, S, L, E, y)
            if s > best + 1e-9:
                best, best_tau = s, tau
        params["knn"]["tau_hard"] = best_tau
    return params


def subset(idx):
    return {t: VAL_LOGITS[t][idx] for t in TAGS}, val_sims[idx], val_nlab[idx], exact_val[idx], y_val[idx]

## 9. Cross-validated stage gains (honest estimate) → auto-disable

In [ ]:
from sklearn.model_selection import StratifiedKFold


def cv_stage_table(disabled):
    skf = StratifiedKFold(TUNE.CV_FOLDS, shuffle=True, random_state=TUNE.SEED)
    rows = []
    for f, (fit_idx, ho_idx) in enumerate(skf.split(np.zeros(len(y_val)), y_val)):
        params = tune_all(*subset(fit_idx), disabled=disabled, seed=f)
        Zh, Sh, Lh, Eh, yh = subset(ho_idx)
        row = {"fold": f}
        for i in range(len(STAGES) + 1):
            name = "baseline" if i == 0 else "+" + STAGES[i - 1]
            row[name] = score(neutralize(params, STAGES[i:]), Zh, Sh, Lh, Eh, yh)
        rows.append(row)
    return pd.DataFrame(rows).set_index("fold")


t0 = time.time()
cv = cv_stage_table(frozenset())
print("held-out macro-F1 per fold, stages applied cumulatively (baseline = equal-weight mean of raw softmax):")
print(cv.round(4).to_string())
gains = cv.diff(axis=1).iloc[:, 1:].mean()
print("\nmean held-out gain per stage:")
print(gains.round(5).to_string())

DISABLED = set()
if TUNE.AUTO_DISABLE:
    DISABLED = {s for s in STAGES if gains["+" + s] < 0}
    if DISABLED:
        print("\nauto-disabled (negative held-out gain):", sorted(DISABLED))
        cv = cv_stage_table(frozenset(DISABLED))
        print(cv.round(4).to_string())

CV_FINAL = float(cv.iloc[:, -1].mean())
CV_BASE = float(cv["baseline"].mean())
print(f"\nCV estimate  baseline {CV_BASE:.4f}  ->  full pipeline {CV_FINAL:.4f}  "
      f"({CV_FINAL - CV_BASE:+.4f})   [{time.time()-t0:.0f}s]")

## 10. Final fit on all of validation

In [ ]:
FINAL_PARAMS = tune_all(VAL_LOGITS, val_sims, val_nlab, exact_val, y_val,
                        disabled=frozenset(DISABLED), seed=TUNE.SEED)

print("in-sample stage scores on full validation (optimistic, CV above is the honest number):")
for i in range(len(STAGES) + 1):
    name = "baseline" if i == 0 else "+" + STAGES[i - 1]
    print(f"  {name:<12} {score(neutralize(FINAL_PARAMS, STAGES[i:]), VAL_LOGITS, val_sims, val_nlab, exact_val, y_val):.4f}")

print("\nmembers:")
for m in FINAL_PARAMS["members"]:
    print(f"  {m['tag']:<18} T={m['temperature']:.3f}  weight={m['weight']:.3f}  params={m['params']/1e6:.1f}M")
print("knn          :", FINAL_PARAMS["knn"])
print("class_weights:", np.round(FINAL_PARAMS["class_weights"], 4))
print("exact_match  :", FINAL_PARAMS["exact_match"])

from sklearn.metrics import classification_report, confusion_matrix
pred_val, prob_val, src_val = final_predict(VAL_LOGITS, val_sims, val_nlab, exact_val, FINAL_PARAMS)
print(classification_report(y_val, pred_val, digits=3, target_names=[CLASS_NAMES[i] for i in range(N_CLASSES)]))
print(confusion_matrix(y_val, pred_val))
print(pd.Series(src_val).value_counts().to_dict())

## 11. Write the production bundle

The memories are rebuilt on train + validation. The tuned weights are left unchanged.

In [ ]:
mem_keys = train_keys + (val_keys if TUNE.PROD_MEMORY_INCLUDES_VAL else [])
mem_y = list(train["y"]) + (list(y_val) if TUNE.PROD_MEMORY_INCLUDES_VAL else [])

lookup_prod = build_lookup(mem_keys, mem_y)
with open(BUNDLE_DIR / "lookup.json", "w", encoding="utf-8") as fh:
    json.dump(lookup_prod, fh, ensure_ascii=False)

KnnIndex().fit(mem_keys, mem_y).save(BUNDLE_DIR / "knn_index.joblib")

total_params = int(sum(m["params"] for m in FINAL_PARAMS["members"]))
bundle = {
    "version": 1,
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "params": FINAL_PARAMS,
    "knn_k_max": TUNE.KNN_K_MAX,
    "bn_normalizer_used": BN_NORMALIZER_AVAILABLE,
    "uses_bangla_family": any(m["family"] == "bangla" for m in FINAL_PARAMS["members"]),
    "lookup_file": "lookup.json",
    "knn_file": "knn_index.joblib",
    "memory_includes_val": TUNE.PROD_MEMORY_INCLUDES_VAL,
    "total_params": total_params,
    "disabled_stages": sorted(DISABLED),
    "scores": {"cv_baseline": CV_BASE, "cv_pipeline": CV_FINAL,
               "val_in_sample": macro_f1(y_val, pred_val)},
    "transformers_version": transformers.__version__,
}
with open(BUNDLE_DIR / "ensemble_config.json", "w") as fh:
    json.dump(bundle, fh, indent=2)

# for later analysis / re-tuning without re-running the models
np.savez_compressed(BUNDLE_DIR / "val_member_logits.npz", **VAL_LOGITS)
pd.DataFrame({"id": val["id"], "y": y_val, "y_pred": pred_val, "source": src_val,
              **{f"p{i}": prob_val[:, i] for i in range(N_CLASSES)}}
             ).to_csv(BUNDLE_DIR / "val_predictions.csv", index=False)
cv.to_csv(BUNDLE_DIR / "cv_stage_scores.csv")

print(f"lookup entries {len(lookup_prod)}   kNN memory rows {len(mem_keys)}   total params {total_params/1e6:.1f}M")
for p in sorted(BUNDLE_DIR.glob("*")):
    print(f"  {p.name:<28} {p.stat().st_size/1e6:8.2f} MB")